# 02 — Exercises: binary classification

The churn examples are synthetic. Complete the written analysis as well as the code. These notebooks use separate training, validation, and final test sets: do not choose a model setting or threshold by looking at final test outcomes.

## Setup

Training data fit the model, validation data support choices, and the final test set is reserved for one assessment after the model and threshold are frozen.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def find_data_file(filename):
    for directory in (Path.cwd(), *Path.cwd().parents):
        data_path = directory / "data" / filename
        if data_path.is_file():
            return data_path
    raise FileNotFoundError(f"Could not find data/{filename}")


customer_data = pd.read_csv(find_data_file("churn.csv"))
feature_data = customer_data.drop(columns="churn")
churn_response = customer_data["churn"].astype(int)

(
    development_features,
    final_test_features,
    development_response,
    final_test_response,
) = train_test_split(
    feature_data,
    churn_response,
    test_size=0.20,
    random_state=19,
    stratify=churn_response,
)
(
    training_features,
    validation_features,
    training_response,
    validation_response,
) = train_test_split(
    development_features,
    development_response,
    test_size=0.25,
    random_state=20,
    stratify=development_response,
)
numeric_columns = ["tenure_months", "monthly_charge", "support_calls"]
categorical_columns = ["contract"]
preprocessing = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            numeric_columns,
        ),
        (
            "category",
            Pipeline(
                steps=[
                    ("impute", SimpleImputer(strategy="most_frequent")),
                    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_columns,
        ),
    ]
)
print("Training class proportions:")
print(training_response.value_counts(normalize=True).sort_index().round(3))
print(f"Training rows: {len(training_features)}")
print(f"Validation rows: {len(validation_features)}")
print(f"Final test rows: {len(final_test_features)}")

## Exercise 1 — Baseline and logistic model

Fit the starter logistic pipeline and report validation accuracy, precision, recall, F1, ROC-AUC, and confusion matrix at threshold 0.5. Compare accuracy with the most-frequent baseline. Why can that baseline look respectable while detecting no churners?

In [ ]:
# Fit the starter logistic model on training rows.
classification_model = Pipeline(
    steps=[
        ("preprocessing", preprocessing),
        ("model", LogisticRegression(max_iter=2000)),
    ]
)
classification_model.fit(training_features, training_response)
validation_probabilities = classification_model.predict_proba(validation_features)[:, 1]
validation_predictions = (validation_probabilities >= 0.5).astype(int)

majority_baseline = DummyClassifier(strategy="most_frequent")
majority_baseline.fit(training_features, training_response)
baseline_predictions = majority_baseline.predict(validation_features)

print("Validation baseline accuracy:", accuracy_score(validation_response, baseline_predictions))
print("Validation confusion matrix:")
print(confusion_matrix(validation_response, validation_predictions))
print("Validation accuracy:", accuracy_score(validation_response, validation_predictions))
validation_precision = precision_score(
    validation_response, validation_predictions, zero_division=0
)
validation_recall = recall_score(
    validation_response, validation_predictions, zero_division=0
)
validation_f1 = f1_score(
    validation_response, validation_predictions, zero_division=0
)
print("Validation precision:", validation_precision)
print("Validation recall:", validation_recall)
print("Validation F1:", validation_f1)
print("Validation ROC-AUC:", roc_auc_score(validation_response, validation_probabilities))

## Exercise 2 — Cost and threshold on validation

Suppose a false positive costs 1 and a missed churn case costs 5, with zero cost for correct decisions. For calibrated probability $p$, derive the cost-optimal rule by comparing $C_{FP}(1-p)$ with $C_{FN}p$. Calculate the theoretical cutoff, then compare candidate thresholds using **validation** rows only. The starter table selects its minimum validation cost and applies that frozen threshold for one final test evaluation. Explain why the test results must not be used to revise the threshold.

In [ ]:
false_positive_cost = 1.0
false_negative_cost = 5.0
threshold_candidates = [0.10, 0.17, 0.25, 0.35, 0.50, 0.65, 0.80]
validation_cost_rows = []

for threshold in threshold_candidates:
    validation_decisions = (validation_probabilities >= threshold).astype(int)
    true_negative, false_positive, false_negative, true_positive = confusion_matrix(
        validation_response, validation_decisions, labels=[0, 1]
    ).ravel()
    validation_cost_rows.append(
        {
            "threshold": threshold,
            "false_positives": false_positive,
            "false_negatives": false_negative,
            "estimated_cost": (false_positive_cost * false_positive
                               + false_negative_cost * false_negative),
        }
    )

validation_cost_table = pd.DataFrame(validation_cost_rows).sort_values(
    ["estimated_cost", "threshold"]
)
print("Choose the threshold from this validation table:")
print(validation_cost_table.to_string(index=False))

# Freeze the validation-selected threshold before the one final test evaluation.
selected_threshold = float(validation_cost_table.iloc[0]["threshold"])
final_test_probabilities = classification_model.predict_proba(final_test_features)[:, 1]
final_test_predictions = (final_test_probabilities >= selected_threshold).astype(int)
print(f"Frozen validation threshold: {selected_threshold:.2f}")
print("Final test confusion matrix:")
print(confusion_matrix(final_test_response, final_test_predictions))
print("Final test ROC-AUC:", roc_auc_score(final_test_response, final_test_probabilities))

## Exercise 3 — Explain metrics

Define precision and recall in words. Give one example where high recall is preferred and one where high precision is preferred. Explain why ROC-AUC does not determine the threshold by itself.

## Exercise 4 — Cross-validation

Compare `C` values `[0.03, 0.1, 0.3, 1, 3, 10]` by stratified five-fold ROC-AUC on training rows. Keep preprocessing inside the pipeline. Update the fitted model with the selected setting, recompute validation probabilities, and repeat your threshold selection. Evaluate the frozen model and threshold on final test data once.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# TODO: compare candidate C values with folds from training data only.
# Keep `preprocessing` inside the pipeline so each fold learns its own transforms.
print("Candidate C values:", [0.03, 0.1, 0.3, 1.0, 3.0, 10.0])
print("Validation is reserved for the threshold choice; test is reserved for final evaluation.")

## Exercise 5 — Limits

Name two reasons why an association between support calls and churn in generated data may fail to guide a real intervention. Consider omitted causes, measurement, privacy, fairness, or behavior change.